# TRY Regime Analysis: From Research to Production

This notebook demonstrates the end-to-end pipeline for analyzing Turkish Lira (TRY) regimes, inferring central bank interventions, and backtesting a trading strategy. 

**Warning:** Trading FX involves significant risk. This notebook is for research purposes.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from src.data.loader import DataLoader
from src.features.builder import FeatureBuilder
from src.models.regime_detector import RegimeDetector
from src.analysis.intervention import InterventionInference
from src.analysis.backtest import Backtester
from src.analysis.risk import RiskManager
from src.analysis.stats import QuantStats
from src.visualization.plots import plot_regime_timeline, plot_intervention_analysis

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 7)

## 1. Data Ingestion
We fetch FX prices (USDTRY), EM FX index, and macro indicators (reserves, rates, inflation).

In [ ]:
loader = DataLoader()
raw_data = loader.get_all_data(cache=True)
print(f"Data Range: {raw_data.index.min()} to {raw_data.index.max()}")
raw_data.tail()

## 2. Feature Engineering
We calculate log returns, rolling volatility, volatility suppression ratios, and trend deviations.

In [ ]:
builder = FeatureBuilder(raw_data)
df = builder.build()
df.tail()

## 3. Walk-Forward Regime Detection
To avoid **lookahead bias**, we use a walk-forward approach where the model for day $T$ is only trained on data available up to day $T-1$.

In [ ]:
detector = RegimeDetector(n_components=3)
regimes = detector.walk_forward_predict(df, train_window=252*2, step=63)
df['regime'] = regimes

# Filter out the initial training period
df_analysis = df[df['regime'] != -1].copy()

print("Regime Characteristics:")
print(detector.get_regime_characteristics(df_analysis, df_analysis['regime']))

In [ ]:
plot_regime_timeline(df_analysis, df_analysis['regime'], title="USDTRY Walk-Forward Regimes")
plt.show()

## 4. Advanced Statistics
We check for stationarity and the half-life of mean reversion for deviations from the long-run trend.

In [ ]:
print(f"FX Returns Stationarity: {QuantStats.test_stationarity(df_analysis['fx_returns'])}")
hl = QuantStats.calculate_half_life(df_analysis['trend_deviation'])
print(f"Half-life of trend deviation: {hl:.2f} days")

## 5. Intervention Inference
We calculate the intervention intensity score based on volatility suppression and reserve dynamics.

In [ ]:
inference = InterventionInference(df_analysis)
df_analysis = inference.calculate_intervention_score()

plot_intervention_analysis(df_analysis)
plt.show()

## 6. Backtesting Strategy
We simulate a strategy that:
- **Shorts USDTRY** (Ears Carry) in Regime 0 (Stability).
- **Stays Flat** in Regime 1 (Moderate Depreciation).
- **Longs USDTRY** in Regime 2 (Panic/Devaluation).

We account for transaction costs (5 bps) and estimated carry.

In [ ]:
backtester = Backtester(df_analysis, initial_capital=100000, transaction_costs=0.0005)
results = backtester.simulate_regime_strategy(carry_rate=0.20) 

plt.figure(figsize=(15, 6))
plt.plot(results.index, results['equity_curve'], label='Strategy Equity Curve')
plt.title("Strategy Performance (Net of Costs & Carry)")
plt.legend()
plt.show()

print("Backtest Metrics:")
print(backtester.get_metrics())

## 7. Risk Management
We analyze the Value-at-Risk (VaR) and Expected Shortfall for each regime.

In [ ]:
risk = RiskManager(results['net_returns'])
print(f"Portfolio Daily VaR (95%): {risk.calculate_var(0.95):.2%}")
print(f"Portfolio Daily ES (95%): {risk.calculate_expected_shortfall(0.95):.2%}")

print("\nRegime-Specific Risk Profile:")
print(json.dumps(risk.calculate_regime_risk(results), indent=4))